In [1]:
! git clone https://github.com/zhengchen1999/DAT.git

Cloning into 'DAT'...
remote: Enumerating objects: 440, done.
remote: Counting objects: 100% (174/174), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 440 (delta 153), reused 141 (delta 139), pack-reused 266 (from 1)
Receiving objects: 100% (440/440), 15.38 MiB | 37.95 MiB/s, done.
Resolving deltas: 100% (255/255), done.


In [2]:
import sys
print(sys.version)

3.10.12 (main, Nov  6 2024, 20:22:13) [GCC 11.4.0]


In [3]:
! pip install -r /kaggle/working/DAT/requirements.txt

ERROR: Could not find a version that satisfies the requirement torch==1.8.0 (from versions: 1.11.0, 1.12.0, 1.12.1, 1.13.0, 1.13.1, 2.0.0, 2.0.1, 2.1.0, 2.1.1, 2.1.2, 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0)
ERROR: No matching distribution found for torch==1.8.0


In [4]:
!pip install tqdm
!pip install yapf
!pip install timm
!pip install einops
!pip install h5py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 10.3 MB/s eta 0:00:00


下载预训练模型

In [5]:
!gdown --folder https://drive.google.com/drive/folders/14VG5mw5ie8RrR4jjypeHynXDZYWL8w-r?usp=sharing -O /kaggle/working/DAT/experiments/pretrained_models/

Retrieving folder contents
Processing file 1AYfLMnIqSlOJyOGabaRI48TEJh440fsN DAT_x2.pth
Processing file 1zRzZl8ogogzCSe6HDtcn4gk0ubH-5-aE DAT_x3.pth
Processing file 1pEhXmg--IWHaZOwHUFdh7TEJqt2qeuYg DAT_x4.pth
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1AYfLMnIqSlOJyOGabaRI48TEJh440fsN
From (redirected): https://drive.google.com/uc?id=1AYfLMnIqSlOJyOGabaRI48TEJh440fsN&confirm=t&uuid=3c3c5040-6be2-4973-8d24-8256dbe7b951
To: /kaggle/working/DAT/experiments/pretrained_models/DAT/DAT_x2.pth
100%|█████████████████████████████████████████| 154M/154M [00:01<00:00, 138MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1zRzZl8ogogzCSe6HDtcn4gk0ubH-5-aE
From (redirected): https://drive.google.com/uc?id=1zRzZl8ogogzCSe6HDtcn4gk0ubH-5-aE&confirm=t&uuid=64975cb1-609d-46ba-a5f9-0cff876ddbbc
To: /kaggle/working/DAT/experiments/pretrained_models/DAT/DAT_x3.pth


下载数据集

In [6]:
!cp -r /kaggle/input/dat-datasets-test /kaggle/working/DAT/datasets

In [7]:
cd /kaggle/working/DAT

/kaggle/working/DAT


In [8]:
%%writefile setup.py

#!/usr/bin/env python

from setuptools import find_packages, setup

import os
import subprocess
import time
import torch
from torch.utils.cpp_extension import BuildExtension, CppExtension, CUDAExtension

version_file = '/kaggle/working/DAT/basicsr/version.py'


def readme():
    with open('README.md', encoding='utf-8') as f:
        content = f.read()
    return content


def get_git_hash():

    def _minimal_ext_cmd(cmd):
        # construct minimal environment
        env = {}
        for k in ['SYSTEMROOT', 'PATH', 'HOME']:
            v = os.environ.get(k)
            if v is not None:
                env[k] = v
        # LANGUAGE is used on win32
        env['LANGUAGE'] = 'C'
        env['LANG'] = 'C'
        env['LC_ALL'] = 'C'
        out = subprocess.Popen(cmd, stdout=subprocess.PIPE, env=env).communicate()[0]
        return out

    try:
        out = _minimal_ext_cmd(['git', 'rev-parse', 'HEAD'])
        sha = out.strip().decode('ascii')
    except OSError:
        sha = 'unknown'

    return sha


def get_hash():
    if os.path.exists('.git'):
        sha = get_git_hash()[:7]
    # currently ignore this
    # elif os.path.exists(version_file):
    #     try:
    #         from basicsr.version import __version__
    #         sha = __version__.split('+')[-1]
    #     except ImportError:
    #         raise ImportError('Unable to get git version')
    else:
        sha = 'unknown'

    return sha


def write_version_py():
    content = """# GENERATED VERSION FILE
# TIME: {}
__version__ = '{}'
__gitsha__ = '{}'
version_info = ({})
"""
    sha = get_hash()
    with open('VERSION', 'r') as f:
        SHORT_VERSION = f.read().strip()
    VERSION_INFO = ', '.join([x if x.isdigit() else f'"{x}"' for x in SHORT_VERSION.split('.')])

    version_file_str = content.format(time.asctime(), SHORT_VERSION, sha, VERSION_INFO)
    with open(version_file, 'w') as f:
        f.write(version_file_str)


def get_version():
    with open(version_file, 'r') as f:
        exec(compile(f.read(), version_file, 'exec'))
    return locals()['__version__']


def make_cuda_ext(name, module, sources, sources_cuda=None):
    if sources_cuda is None:
        sources_cuda = []
    define_macros = []
    extra_compile_args = {'cxx': []}

    if torch.cuda.is_available() or os.getenv('FORCE_CUDA', '0') == '1':
        define_macros += [('WITH_CUDA', None)]
        extension = CUDAExtension
        extra_compile_args['nvcc'] = [
            '-D__CUDA_NO_HALF_OPERATORS__',
            '-D__CUDA_NO_HALF_CONVERSIONS__',
            '-D__CUDA_NO_HALF2_OPERATORS__',
        ]
        sources += sources_cuda
    else:
        print(f'Compiling {name} without CUDA')
        extension = CppExtension

    return extension(
        name=f'{module}.{name}',
        sources=[os.path.join(*module.split('.'), p) for p in sources],
        define_macros=define_macros,
        extra_compile_args=extra_compile_args)


def get_requirements(filename='requirements.txt'):
    here = os.path.dirname(os.path.realpath(__file__))
    with open(os.path.join(here, filename), 'r') as f:
        requires = [line.replace('\n', '') for line in f.readlines()]
    return requires


if __name__ == '__main__':
    cuda_ext = os.getenv('BASICSR_EXT')  # whether compile cuda ext
    if cuda_ext == 'True':
        ext_modules = [
            make_cuda_ext(
                name='deform_conv_ext',
                module='basicsr.ops.dcn',
                sources=['src/deform_conv_ext.cpp'],
                sources_cuda=['src/deform_conv_cuda.cpp', 'src/deform_conv_cuda_kernel.cu']),
            make_cuda_ext(
                name='fused_act_ext',
                module='basicsr.ops.fused_act',
                sources=['src/fused_bias_act.cpp'],
                sources_cuda=['src/fused_bias_act_kernel.cu']),
            make_cuda_ext(
                name='upfirdn2d_ext',
                module='basicsr.ops.upfirdn2d',
                sources=['src/upfirdn2d.cpp'],
                sources_cuda=['src/upfirdn2d_kernel.cu']),
        ]
    else:
        ext_modules = []

    write_version_py()
    setup(
        name='basicsr',
        version=get_version(),
        description='Open Source Image and Video Super-Resolution Toolbox',
        long_description=readme(),
        long_description_content_type='text/markdown',
        author='Xintao Wang',
        author_email='xintao.wang@outlook.com',
        keywords='computer vision, restoration, super resolution',
        url='https://github.com/xinntao/BasicSR',
        include_package_data=True,
        packages=find_packages(exclude=('options', 'datasets', 'experiments', 'results', 'tb_logger', 'wandb')),
        classifiers=[
            'Development Status :: 4 - Beta',
            'License :: OSI Approved :: Apache Software License',
            'Operating System :: OS Independent',
            'Programming Language :: Python :: 3',
            'Programming Language :: Python :: 3.7',
            'Programming Language :: Python :: 3.8',
        ],
        license='Apache License 2.0',
        setup_requires=['cython', 'numpy'],
        install_requires=get_requirements(),
        ext_modules=ext_modules,
        cmdclass={'build_ext': BuildExtension},
        zip_safe=False)

Overwriting setup.py


In [9]:
%%writefile /kaggle/working/DAT/options/Test/test_DAT_x4.yml
# general settings
name: test_DAT_x4
model_type: DATModel
scale: 4
num_gpu: 2
manual_seed: 10

datasets:
  test_1:  # the 1st test dataset
    task: SR
    name: Set5
    type: PairedImageDataset
    dataroot_gt: /kaggle/working/DAT/datasets/dat-datasets-test/benchmark/Set5/HR
    dataroot_lq: /kaggle/working/DAT/datasets/dat-datasets-test/benchmark/Set5/LR_bicubic/X4
    filename_tmpl: '{}x4'
    io_backend:
      type: disk

  test_2:  # the 2st test dataset
    task: SR
    name: Set14
    type: PairedImageDataset
    dataroot_gt: /kaggle/working/DAT/datasets/dat-datasets-test/benchmark/Set14/HR
    dataroot_lq: /kaggle/working/DAT/datasets/dat-datasets-test/benchmark/Set14/LR_bicubic/X4
    filename_tmpl: '{}x4'
    io_backend:
      type: disk

  test_3:  # the 3st test dataset
    task: SR
    name: B100
    type: PairedImageDataset
    dataroot_gt: /kaggle/working/DAT/datasets/dat-datasets-test/benchmark/B100/HR
    dataroot_lq: /kaggle/working/DAT/datasets/dat-datasets-test/benchmark/B100/LR_bicubic/X4
    filename_tmpl: '{}x4'
    io_backend:
      type: disk

  test_4:  # the 4st test dataset
    task: SR
    name: Urban100
    type: PairedImageDataset
    dataroot_gt: /kaggle/working/DAT/datasets/dat-datasets-test/benchmark/Urban100/HR
    dataroot_lq: /kaggle/working/DAT/datasets/dat-datasets-test/benchmark/Urban100/LR_bicubic/X4
    filename_tmpl: '{}x4'
    io_backend:
      type: disk

  test_5:  # the 5st test dataset
    task: SR
    name: Manga109
    type: PairedImageDataset
    dataroot_gt: /kaggle/working/DAT/datasets/dat-datasets-test/benchmark/Manga109/HR
    dataroot_lq: /kaggle/working/DAT/datasets/dat-datasets-test/benchmark/Manga109/LR_bicubic/X4
    filename_tmpl: '{}_LRBI_x4'
    io_backend:
      type: disk


# network structures
network_g:
  type: DAT
  upscale: 4
  in_chans: 3
  img_size: 64
  img_range: 1.
  split_size: [8,32]
  depth: [6,6,6,6,6,6]
  embed_dim: 180
  num_heads: [6,6,6,6,6,6]
  expansion_factor: 4
  resi_connection: '1conv'

# path
path:
  pretrain_network_g: /kaggle/working/DAT/experiments/pretrained_models/DAT/DAT_x4.pth
  strict_load_g: True

# validation settings
val:
  save_img: True
  suffix: ~  # add suffix to saved images, if None, use exp name
  use_chop: False  # True to save memory, if img too large

  metrics:
    psnr: # metric name, can be arbitrary
      type: calculate_psnr
      crop_border: 4
      test_y_channel: True
    ssim:
      type: calculate_ssim
      crop_border: 4
      test_y_channel: True


Overwriting /kaggle/working/DAT/options/Test/test_DAT_x4.yml


In [10]:
!python /kaggle/working/DAT/setup.py develop

/usr/local/lib/python3.10/dist-packages/setuptools/__init__.py:94: _DeprecatedInstaller: setuptools.installer and fetch_build_eggs are deprecated.
!!

        ********************************************************************************
        Requirements should be satisfied by a PEP 517 installer.
        If you are using pip, you can try `pip install --use-pep517`.
        ********************************************************************************

!!
  dist.fetch_build_eggs(dist.setup_requires)
/usr/local/lib/python3.10/dist-packages/setuptools/command/develop.py:41: EasyInstallDeprecationWarning: easy_install command is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` and ``easy_install``.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://github.com/pypa/setuptools/issues/917 for details.
        *********************

In [11]:
!pip3 install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu118

Looking in indexes: https://download.pytorch.org/whl/nightly/cu118


In [12]:
import torch
print("PyTorch 版本:", torch.__version__)
print("CUDA 是否可用:", torch.cuda.is_available())
print("GPU 数量:", torch.cuda.device_count())
print("当前 GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "无 GPU")

PyTorch 版本: 2.5.1+cu121
CUDA 是否可用: True
GPU 数量: 2
当前 GPU: Tesla T4


In [13]:
!python /kaggle/working/DAT/basicsr/test.py -opt /kaggle/working/DAT/options/Test/test_DAT_x4.yml

/usr/local/lib/python3.10/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Disable distributed.
2025-02-25 06:14:31,042 INFO: 
  name: test_DAT_x4
  model_type: DATModel
  scale: 4
  num_gpu: 2
  manual_seed: 10
  datasets:[
    test_1:[
      task: SR
      name: Set5
      type: PairedImageDataset
      dataroot_gt: /kaggle/working/DAT/datasets/dat-datasets-test/benchmark/Set5/HR
      dataroot_lq: /kaggle/working/DAT/datasets/dat-datasets-test/benchmark/Set5/LR_bicubic/X4
      filename_tmpl: {}x4
      io_backend:[
        type: disk
      ]
      phase: test
      scale: 4
    ]
    test_2:[
      task: SR
      name: Set14
      type: PairedImageDataset
      dataroot_gt: /kaggle/working/DAT/datasets/dat-datasets-test/benchmark/Set14/HR
      dataroot_lq: /kaggle/working/DAT/datasets